# Data Collection

**Progetto**: GDA Freshness & Waste Explorer 

**Dataset** :[Retail Store Sales: Dirty for Data Cleaning](https://www.kaggle.com/datasets/ahmedmohamed2003/retail-store-sales-dirty-for-data-cleaning)

Con questo notebook puoi scaricare il daset direttamente da kaggle (senza il download manualew del file), caricarlo in un DataFrame pandas e fare una prima verifica strutturale 

**Nota sull'autenticazione Kaggle:** per eseguire la cella di download serve un account Kaggle gratuito e il
file `kaggle.json` (Kaggle → Settings → API → Create New Token) salvato in `~/.kaggle/kaggle.json`.

In [1]:
#Scaricamento del dataset 
import kagglehub
import pandas as pd
import os

# Scarica (o usa la cache locale se già presente) l'ultima versione del dataset
path = kagglehub.dataset_download("ahmedmohamed2003/retail-store-sales-dirty-for-data-cleaning")

print("Dataset scaricato in:", path)
print("File presenti:", os.listdir(path))

# Trova il CSV dentro la cartella scaricata e caricalo
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))

Dataset scaricato in: C:\Users\giuse\.cache\kagglehub\datasets\ahmedmohamed2003\retail-store-sales-dirty-for-data-cleaning\versions\1
File presenti: ['retail_store_sales.csv']


In [2]:
df.head()


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [3]:
# Tipo di dato 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.9+ MB


**NOTE**:Quasi tutte le colonne hanno il formato corretto . Solo la colonna 'Transaction Date' ha il formato stringa , ma è necesssario convertirla nel formato corretto

In [4]:
df['Transaction Date']=pd.to_datetime(df['Transaction Date'])

In [5]:
df.dtypes


Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied            object
dtype: object

Estraggo mesi, giorni, anni 

In [6]:
df['Year']= df['Transaction Date'].dt.year
df['Month']= df['Transaction Date'].dt.month
df['Day_name']= df['Transaction Date'].dt.day_name()

In [7]:
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Year,Month,Day_name
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4,Monday
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7,Sunday
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10,Wednesday
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN,2022,5,Saturday
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10,Sunday


Verifica dei valori mancanti 

In [8]:
#Verifica dei valori nulli 
missing= df.isna().sum().to_frame('n_missing')

missing['%_missing']= (missing['n_missing']/len(df)*100).round(2)

missing.sort_values('n_missing',ascending=False)

,n_missing,%_missing
Discount Applied,4199,33.39
Item,1213,9.65
Price Per Unit,609,4.84
Quantity,604,4.80
Total Spent,604,4.80
Transaction ID,0,0.00
Category,0,0.00
Customer ID,0,0.00
Location,0,0.00
Payment Method,0,0.00


**NOTE**:Abbiamo diverse colonne (Price Per Unit, Item, Quantity, Total Spent, Discount Applied) con valori nulli 

Verifica dei valori duplicati

In [9]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
12570    False
12571    False
12572    False
12573    False
12574    False
Length: 12575, dtype: bool

**NOTE**: Non ci sono valori duplicati 

Elenco dei prodotti presenti. Verifico se la categoria 'Butchers' è presente 

In [10]:
df.columns

Index(['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit',
       'Quantity', 'Total Spent', 'Payment Method', 'Location',
       'Transaction Date', 'Discount Applied', 'Year', 'Month', 'Day_name'],
      dtype='str')

In [11]:
df['Category'].value_counts()

Category
Furniture                             1591
Electric household essentials         1591
Food                                  1588
Milk Products                         1584
Butchers                              1568
Beverages                             1567
Computers and electric accessories    1558
Patisserie                            1528
Name: count, dtype: int64

**NOTE**: La categoria 'Butchers' è presente 

In [12]:
#Salvataggio del nuovo file csv aggiornato 
df.to_csv('df_aggiornato.csv', index=False)